# Pre-training on pop-cosmos: Cardinal Y1 complete with photometric errors propagated

The exercise of `CNN_photoz-ensemble-cardinal_noise_realizations.ipynb` (noisy Cardinal, Y1 depth,
i < 23, all 9 LSST+Roman bands; p(z) from `znn.ensemble_predict_resampled`), now testing **pre-training
on another survey**: the ensemble trained on the pop-cosmos mock in its native COSMOS2020 bands
(`CNN_photoz-ensemble-popcosmos.ipynb`). Both catalogues are encoded on the same wavelength grid
(`lambda_range` = the LSST+Roman span), so a model trained on one takes the other's inputs as they are.

Ensembles compared, all with S = `N_SAMPLES` noise realizations at prediction time:
- **from scratch**: trained on Cardinal only (the noise-realizations notebook's ensemble, reloaded);
- **pre-trained only**: the pop-cosmos ensemble applied to Cardinal with no Cardinal training;
- **pre-trained + fine-tuned**: the pop-cosmos ensemble fine-tuned on the Cardinal training set with
  `znn.fine_tune_pre_trained_model` (first 4 layers frozen, learning rate 1e-4, one fold per member).

Caveats: the mock's photometry is COSMOS2020-depth, far less noisy than LSST Y1 in the optical, and its
galaxy population differs from Cardinal's; the "pre-trained only" result mixes both effects.

In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from qp.metrics.pit import PIT

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

import znn  # run from the repo root so this picks up the local source

I0000 00:00:1791378078.171540  533862 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791378078.172647  533862 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791378078.272367  533862 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791378081.108267  533862 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

## Configuration

In [2]:
# Parametric paths (edit via environment variables, or defaults below)
CARDINAL_DATA_ROOT = os.environ.get("CNNPZ_CARDINAL_DATA_ROOT", "data")
FILTER_ROOT = os.environ.get("CNNPZ_FILTER_ROOT", "data")
MODEL_ROOT = os.environ.get("CNNPZ_MODEL_ROOT", "models/noise_realizations")

TRAIN_FILE = os.path.join(CARDINAL_DATA_ROOT, "train_100k_noisy_y1_i23.parquet")
TEST_FILE = os.path.join(CARDINAL_DATA_ROOT, "test_100k_noisy_y1_i23.parquet")

# band -> (catalogue magnitude column, filter curve file), ordered by wavelength. The Roman columns
# are read under their original names, so no renaming is needed.
BANDS = {
    "u": ("mag_u_lsst", os.path.join(FILTER_ROOT, "DC2LSST_u.res")),
    "g": ("mag_g_lsst", os.path.join(FILTER_ROOT, "DC2LSST_g.res")),
    "r": ("mag_r_lsst", os.path.join(FILTER_ROOT, "DC2LSST_r.res")),
    "i": ("mag_i_lsst", os.path.join(FILTER_ROOT, "DC2LSST_i.res")),
    "z": ("mag_z_lsst", os.path.join(FILTER_ROOT, "DC2LSST_z.res")),
    "y": ("mag_y_lsst", os.path.join(FILTER_ROOT, "DC2LSST_y.res")),
    "Y": ("Roman_obs_Y106", os.path.join(FILTER_ROOT, "roman_Y106.res")),
    "J": ("Roman_obs_J129", os.path.join(FILTER_ROOT, "roman_J129.res")),
    "H": ("Roman_obs_H158", os.path.join(FILTER_ROOT, "roman_H158.res")),
}
REF_BAND = "i"  # magnitudes are normalized by this band

# The pop-cosmos grid: the LSST+Roman span. It is also these bands' own span, so the Cardinal
# inputs are the same as with the default config.
CONFIG = {**znn.DEFAULT_FEATURE_CONFIG, "lambda_range": (3000.0, 18650.0)}

PRETRAINED_FILE = os.environ.get("CNNPZ_POPCOSMOS_MODEL", "models/popcosmos/popcosmos_ensemble_CNN_6layers.pkl")
FINETUNED_DIR = os.path.join(MODEL_ROOT, "y1_complete_finetuned_from_popcosmos_CNN_6layers")

# magnitude column -> photometric error column, used to draw the noise realizations
ERRORS = {column: f"{column}_err" for column, _ in BANDS.values()}

N_SPLITS = 5  # ensemble members (K-fold)
EPOCHS = 100  # max epochs per member (early stopping usually ends sooner)
N_SAMPLES = 50  # noise realizations S at prediction time
NOISE_SEED = 42

## Load training and test data

In [3]:
training_y1 = znn.read_catalog(TRAIN_FILE)
test_y1 = znn.read_catalog(TEST_FILE)

X_y1, Y_y1 = znn.catalog_to_XY(training_y1, BANDS, REF_BAND, CONFIG)
X_test_y1, Y_test_y1 = znn.catalog_to_XY(test_y1, BANDS, REF_BAND, CONFIG)
mag_i_test = test_y1["mag_i_lsst"].to_numpy()
print("train:", X_y1.shape, "test:", X_test_y1.shape)

train: (87088, 32, 3) test: (87390, 32, 3)


## The three ensembles

- **from scratch**: loaded from the noise-realizations notebook's save directory (trained there if missing).
- **pre-trained only**: loaded from `PRETRAINED_FILE`; its stored config must use the same grid.
- **pre-trained + fine-tuned**: fine-tuning changes the models in place, so it starts from a fresh load of
  `PRETRAINED_FILE`. Saved under `FINETUNED_DIR` and reloaded on re-runs.

In [4]:
scratch_dir = os.path.join(MODEL_ROOT, "y1_complete_point_ensemble_CNN_6layers")
if os.path.exists(os.path.join(scratch_dir, "norm_params.json")):
    scratch_models = znn.load_ensemble(save_dir=scratch_dir)
else:
    scratch_models, _ = znn.train_ensembles(znn.build_model, X_y1, Y_y1, N_SPLITS=N_SPLITS, EPOCHS=EPOCHS)
    znn.save_ensemble(scratch_models, save_dir=scratch_dir)

pretrained_models, pretrained_features = znn.load_ensemble_file(PRETRAINED_FILE)
assert pretrained_features["config"] == CONFIG, "the pre-trained ensemble uses another grid or feature config"
print(f"pre-trained on pop-cosmos: {len(pretrained_models)} members, {len(pretrained_features['bands'])} bands")

if os.path.exists(os.path.join(FINETUNED_DIR, "norm_params.json")):
    finetuned_models = znn.load_ensemble(save_dir=FINETUNED_DIR)
    finetune_histories = None
else:
    fresh_pretrained, _ = znn.load_ensemble_file(PRETRAINED_FILE)
    finetuned_models, finetune_histories = znn.fine_tune_pre_trained_model(X_y1, Y_y1, pretrained_models=fresh_pretrained)
    znn.save_ensemble(finetuned_models, save_dir=FINETUNED_DIR)

if finetune_histories is not None:
    znn.plot_ensemble_losses(finetune_histories, ylim=(0, 0.001))

ENSEMBLES = {
    "from scratch": scratch_models,
    "pre-trained only": pretrained_models,
    "pre-trained + fine-tuned": finetuned_models,
}
FOCUS = "pre-trained + fine-tuned"

E0000 00:00:1791378082.355065  533862 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Loaded fold 1 model from models/noise_realizations/y1_complete_point_ensemble_CNN_6layers/model_fold_1.keras
Loaded fold 2 model from models/noise_realizations/y1_complete_point_ensemble_CNN_6layers/model_fold_2.keras
Loaded fold 3 model from models/noise_realizations/y1_complete_point_ensemble_CNN_6layers/model_fold_3.keras
Loaded fold 4 model from models/noise_realizations/y1_complete_point_ensemble_CNN_6layers/model_fold_4.keras
Loaded fold 5 model from models/noise_realizations/y1_complete_point_ensemble_CNN_6layers/model_fold_5.keras


FileNotFoundError: [Errno 2] No such file or directory: 'models/popcosmos/popcosmos_ensemble_CNN_6layers.pkl'

## Predictions on the test set

Every ensemble with S = `N_SAMPLES` noise realizations of the test photometry (the same seed for each).
`width` is the standard deviation of each object's p(z); `components` holds its split into model and
photometric parts.

In [ ]:
ids = test_y1["galaxy_id"].to_numpy()
pz, components = {}, {}
for name, models_ in ENSEMBLES.items():
    pz[name], components[name] = znn.ensemble_predict_resampled(
        models_,
        test_y1,
        ERRORS,
        N_SAMPLES,
        np.random.default_rng(NOISE_SEED),
        BANDS,
        REF_BAND,
        config=CONFIG,
        ids=ids,
        return_components=True,
    )

VERSIONS = list(pz)
z_point, width = {}, {}
for name in VERSIONS:
    z_point[name] = np.squeeze(pz[name].ancil["zmode"])
    width[name] = znn.predictions_mean_std(pz[name])[1].ravel()
    print(f"{name}: median p(z) width = {np.median(width[name]):.4f}")

## Where does the width come from?

For the fine-tuned ensemble: median model and photometric parts of the width / (1 + z), in bins of i magnitude and true redshift, against the actual scatter of Δz (biweight). The photometric part should grow towards faint magnitudes.

In [ ]:
redshift_bins = np.linspace(0, 2.5, 11)
imag_bins = np.linspace(18, 25.5, 11)


def binned_median(x, values, bins):
    idx = np.digitize(x, bins) - 1
    return np.array([np.median(values[idx == i]) if np.any(idx == i) else np.nan for i in range(len(bins) - 1)])


stats_focus = znn.get_all_stats(
    Y_test_y1, z_point[FOCUS], mag_i_test, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins
)

fig, axarr = plt.subplots(1, 2, figsize=[13, 5])
for ax, x, bins, binned, xlabel in [
    (axarr[0], mag_i_test, imag_bins, stats_focus[2], "i magnitude"),
    (axarr[1], Y_test_y1, redshift_bins, stats_focus[1], "true redshift"),
]:
    centres = (bins[1:] + bins[:-1]) / 2
    for part, style in [("model", "-o"), ("photometric", "-s")]:
        ax.plot(centres, binned_median(x, components[FOCUS][part] / (1 + Y_test_y1), bins), style, label=f"{part} part")
    ax.plot(centres, binned_median(x, width[FOCUS] / (1 + Y_test_y1), bins), "-^", color="k", label="total width")
    ax.plot(centres, [s[2] for s in binned], "k--", label="actual scatter")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"$\sigma / (1+z)$")
    ax.set_yscale("log")
    ax.legend(fontsize=10)
plt.suptitle(FOCUS, y=1.02)
plt.tight_layout()
plt.show()

## Point estimates

Δz = (z_phot − z_true) / (1 + z_true), using the p(z) mean as z_phot. It is the mean over the noisy photometry realizations.

In [ ]:
stats = {}
for name in VERSIONS:
    stats[name] = znn.get_all_stats(
        Y_test_y1, z_point[name], mag_i_test, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins
    )
    znn.plot_stats(*stats[name], Y_test_y1, z_point[name], redshift_bins, imag_bins, mag_i_test)
    plt.suptitle(name, y=1.02)
    plt.show()

Fine-tuned against from scratch: solid lines are from scratch, dashed lines fine-tuned.

In [ ]:
display(Markdown(znn.stats_to_markdown(stats["from scratch"][0], stats[FOCUS][0], data_title=("from scratch", FOCUS))))
znn.compare_binned_stats(
    redshift_bins, imag_bins, stats["from scratch"][1], stats["from scratch"][2], stats[FOCUS][1], stats[FOCUS][2]
)
plt.suptitle(f"from scratch (solid) vs {FOCUS} (dashed)", y=1.02)
plt.show()

## p(z) calibration: PIT

PIT = CDF of each object's p(z) at its true redshift. For calibrated p(z) the PIT values are uniform: a flat histogram and a Q–Q curve on the diagonal. A U shape means p(z) too narrow, a hump means too wide. KS, CvM and Anderson–Darling measure the distance from uniform (lower is better). `outlier` is the fraction of PIT values in the extreme tails (< 0.0001 or > 0.9999).

In [ ]:
pit_values, pit_stats = {}, {}
for name in VERSIONS:
    pit = PIT(pz[name], Y_test_y1)
    pit_values[name] = pit.pit_samps
    pit_stats[name] = {
        "ks": pit.evaluate_PIT_KS().statistic,
        "CvM": pit.evaluate_PIT_CvM().statistic,
        "ksamp": pit.evaluate_PIT_anderson_ksamp().statistic,
        "outlier": np.mean((pit.pit_samps < 0.0001) | (pit.pit_samps > 0.9999)),
    }

fig, axarr = plt.subplots(1, 2, figsize=[13, 5])
q = np.linspace(0, 1, 101)
for name in VERSIONS:
    axarr[0].hist(pit_values[name], bins=50, range=(0, 1), density=True, histtype="step", lw=2, label=name)
    axarr[1].plot(q, np.quantile(pit_values[name], q), lw=2, label=name)
axarr[0].axhline(1, color="k", ls="--")
axarr[0].set_xlabel("PIT")
axarr[0].set_ylabel("density")
axarr[0].legend(fontsize=10)
axarr[1].plot([0, 1], [0, 1], "k--")
axarr[1].set_xlabel("uniform quantile")
axarr[1].set_ylabel("PIT quantile")
plt.tight_layout()
plt.show()

pd.DataFrame(pit_stats).round(4)

## Do the widths match the actual errors?

Left and middle: median p(z) width / (1 + z) in bins of i magnitude and true redshift, against the actual scatter of Δz (biweight, from `get_all_stats`). For calibrated p(z) the two should track each other. Right: residuals divided by the predicted width, which should follow a unit Gaussian.

In [ ]:
fig, axarr = plt.subplots(1, 3, figsize=[18, 5])
for k, name in enumerate(VERSIONS):
    rel_width = width[name] / (1 + Y_test_y1)
    for ax, x, bins, binned in [
        (axarr[0], mag_i_test, imag_bins, stats[name][2]),
        (axarr[1], Y_test_y1, redshift_bins, stats[name][1]),
    ]:
        centres = (bins[1:] + bins[:-1]) / 2
        ax.plot(centres, binned_median(x, rel_width, bins), "-o", color=f"C{k}", label=f"{name}: median width")
        ax.plot(centres, [s[2] for s in binned], "--", color=f"C{k}", label=f"{name}: actual scatter")
    axarr[2].hist(
        (z_point[name] - Y_test_y1) / width[name], bins=np.linspace(-6, 6, 121), density=True, histtype="step", lw=2, label=name
    )

zz = np.linspace(-6, 6, 200)
axarr[2].plot(zz, np.exp(-0.5 * zz**2) / np.sqrt(2 * np.pi), "k--", label="unit Gaussian")
axarr[0].set_xlabel("i magnitude")
axarr[1].set_xlabel("true redshift")
for ax in axarr[:2]:
    ax.set_ylabel(r"$\sigma / (1+z)$")
    ax.set_yscale("log")
    ax.legend(fontsize=8)
axarr[2].set_xlabel("(z_phot - z_true) / width")
axarr[2].legend(fontsize=9)
plt.tight_layout()
plt.show()

## Example p(z)

A few random test objects: each version's p(z), with the true redshift as a dashed line.

In [ ]:
rng = np.random.default_rng(1)
examples = rng.choice(len(Y_test_y1), size=8, replace=False)
z_grid = np.linspace(0, 3, 601)

fig, axarr = plt.subplots(2, 4, figsize=[18, 7])
for ax, i in zip(axarr.ravel(), examples):
    for k, name in enumerate(VERSIONS):
        ax.plot(z_grid, np.squeeze(pz[name][int(i)].pdf(z_grid)), color=f"C{k}", label=name)
    ax.axvline(Y_test_y1[i], color="k", ls="--")
    ax.set_xlim(max(0, Y_test_y1[i] - 0.6), Y_test_y1[i] + 0.6)
    ax.set_title(f"i = {mag_i_test[i]:.1f}, z = {Y_test_y1[i]:.2f}", fontsize=11)
axarr[0, 0].legend(fontsize=8)
for ax in axarr[1]:
    ax.set_xlabel("z")
plt.tight_layout()
plt.show()

## Summary

In [ ]:
summary = {}
for name in VERSIONS:
    bias, bias_err, scatter, outlier_rate, abs_outlier_rate = stats[name][0]
    summary[name] = {
        "bias": bias,
        "scatter": scatter,
        "outlier rate (3 sigma)": outlier_rate,
        "outlier rate (|dz| > 0.2)": abs_outlier_rate,
        "median p(z) width": np.median(width[name]),
        **{f"PIT {k}": v for k, v in pit_stats[name].items()},
    }
pd.DataFrame(summary).round(4)